# note分析シート：Colab で取得する（予備）

GitHub Actions と同じ取得プログラムを、**手で1回だけ**動かすためのノートブックです。取ったデータは、シートの「受け取り用ウェブアプリ」へ**署名つき**で送ります（Google ドライブ全体の権限は使いません）。

**はじめに（安全のため）**
- Cookie（`_note_session_v5`）と「受け取り用の合言葉」は、実行中に出る入力欄（`getpass`）に貼ってください。**セルに直接書かない・画面に print しない**でください。入力した値はノートブックに保存されません。
- このノートブックを「ドライブにコピーを保存」して使う場合も、出力に秘密は出しません。念のため「編集 → ノートブックの設定 → 保存時にコードセルの出力を除外する」をオンにするとさらに安心です。
- シートの「設定」で **取得方法 = GitHub** にしてから使います（Colab でも同じ受け取り口を使います）。シート側の自動取得は止まり、二重アクセスになりません。
- note へのアクセスは 3秒以上あけ、403/429 が出たらその場で止めて記録します。**何度も続けて実行しないでください**（1日1回まで）。

**手順**：上から順に ▶ を押す → 最後のセルで入力欄が出たら貼る → 「シートに送りました」と出たら完了。

## 1. 取得プログラムを用意する（中身を見るだけで、まだ note にはアクセスしません）

In [ ]:
import os
os.makedirs('/content/note_runner_pkg/note_runner', exist_ok=True)
print('準備OK')

In [ ]:
%%writefile /content/note_runner_pkg/note_runner/__init__.py
"""note分析シートの外部取得ランナー（GitHub Actions / Google Colab 用）。標準ライブラリだけで動きます。"""
VERSION = "1.3.3"

In [ ]:
%%writefile /content/note_runner_pkg/note_runner/core.py
"""note の応答を読む・URL・安全チェック（ネットワークには触れない純粋関数）。
Apps Script 版（src/na_core.js・src/na_dash.js）と同じ規則・同じ項目です。"""
import json
import re
from datetime import datetime, timedelta, timezone

JST = timezone(timedelta(hours=9))
HOUR_MS = 3600 * 1000
DAY_MS = 24 * HOUR_MS
API = "https://note.com/api"
COOKIE_NAME = "_note_session_v5"
HOST_COOKIE = "note.com"           # Cookie を付けてよい唯一のホスト
HOST_GQL = "graphql.note.com"      # 一時トークン（Bearer）を付けてよい唯一のホスト
STATS_PV_URL = "https://note.com/api/v1/stats/pv"
GQL_AUTH_URL = "https://note.com/api/v3/graphql/auth"
GQL_URL = "https://graphql.note.com/graphql"
METHOD_STATS = "自動（Cookie・stats/pv）"
METHOD_GQL = "自動（Cookie・新ダッシュボード）"
ID_RE = re.compile(r"^[A-Za-z0-9_\-]{1,50}$")
KEY_RE = re.compile(r"^n[0-9a-z]{6,24}$")


class NaError(Exception):
    def __init__(self, code, message):
        super().__init__(message)
        self.code = code
        self.message = message


def s(v):
    return "" if v is None else str(v).strip()


def num(v):
    try:
        x = float(re.sub(r"[,，\s円¥￥]", "", s(v)) or "nan")
    except ValueError:
        return 0
    if x != x or x in (float("inf"), float("-inf")):
        return 0
    return int(x) if x.is_integer() else x


def num0(v):
    return num(v)


def normalize_id(v):
    t = s(v)
    m = re.search(r"note\.com/([A-Za-z0-9_\-]+)", t)
    if m:
        t = m.group(1)
    t = t.lstrip("@")
    return t if ID_RE.match(t) else ""


def jst(ms):
    d = datetime.fromtimestamp(ms / 1000, JST)
    return {"date": d.strftime("%Y-%m-%d"), "time": d.strftime("%H:%M"), "weekday": (d.weekday() + 1) % 7, "hour": d.hour, "stamp": d.strftime("%Y-%m-%d %H:%M")}


def day_start_ms(ms):
    return (ms + 9 * HOUR_MS) // DAY_MS * DAY_MS - 9 * HOUR_MS


def parse_time(v):
    """ISO 形式（+09:00 など）。タイムゾーンが無いものは日本時間として扱う。"""
    t = s(v)
    if not t:
        return None
    try:
        d = datetime.fromisoformat(t.replace("Z", "+00:00"))
    except ValueError:
        return None
    if d.tzinfo is None:
        d = d.replace(tzinfo=JST)
    return int(d.timestamp() * 1000)


# ---------- URL ----------
def quote(x):
    from urllib.parse import quote as q
    return q(str(x), safe="")


def list_url(cid, page):
    return f"{API}/v2/creators/{quote(cid)}/contents?kind=note&page={page}"


def creator_url(cid):
    return f"{API}/v2/creators/{quote(cid)}"


def detail_url(key):
    return f"{API}/v3/notes/{quote(key)}"


def likes_url(key, page):
    return f"{API}/v3/notes/{quote(key)}/likes?page={page}"


def comments_url(key, page):
    """記事のコメント一覧（公開・Cookie なし）。返ってくるのは「親コメント」だけで、新しい順（2026-10 時点で確認）"""
    return f"{API}/v3/notes/{quote(key)}/note_comments?page={page}"


def url_host(url):
    """https のみ。ユーザー名@・ポート指定・バックスラッシュは空文字（＝拒否）"""
    m = re.match(r"^https://([^/?#]*)(?:[/?#]|$)", s(url), re.I)
    if not m:
        return ""
    auth = m.group(1)
    if not auth or "@" in auth or ":" in auth or "\\" in auth:
        return ""
    return auth.lower().rstrip(".")


def key_from_url(u):
    m = re.search(r"/n/(n[0-9a-z]+)", s(u), re.I)
    return m.group(1) if m else ""


# ---------- 秘密の扱い ----------
def normalize_cookie(inp):
    t = re.sub(r"^cookie:\s*", "", s(inp), flags=re.I).replace("\r", "").replace("\n", "")
    if not t:
        return None, "Cookie が空です。"
    v = ""
    if "=" in t:
        for part in t.split(";"):
            if "=" in part:
                k, _, val = part.partition("=")
                if k.strip() == COOKIE_NAME:
                    v = val.strip()
        if not v:
            return None, f"{COOKIE_NAME} が見つかりませんでした。値だけを貼り付けてください。"
    else:
        v = t
    v = v.strip('"')
    if not re.match(r"^[A-Za-z0-9%_\-\.~+/=]{16,4096}$", v):
        return None, "Cookie の値の形が想定と違います（空白や日本語が入っていないか確認してください）。"
    return v, ""


def mask(v):
    t = s(v)
    if not t:
        return ""
    return "****" + (t[-4:] if len(t) > 8 else "")


# ---------- 公開データ ----------
def parse_creator(j):
    d = (j or {}).get("data") if isinstance(j, dict) else None
    if not isinstance(d, dict) or not d.get("urlname"):
        raise NaError("PARSE", "クリエイター情報を読み取れませんでした（noteの仕様が変わった可能性があります）。")
    return {"creator": d["urlname"], "name": s(d.get("nickname")), "followers": num0(d.get("followerCount")), "following": num0(d.get("followingCount")), "noteCount": num0(d.get("noteCount"))}


def parse_list(j, creator_id):
    d = (j or {}).get("data") if isinstance(j, dict) else None
    if not isinstance(d, dict) or not isinstance(d.get("contents"), list):
        raise NaError("PARSE", "記事一覧を読み取れませんでした（noteの仕様が変わった可能性があります）。")
    out = []
    for c in d["contents"]:
        ms = parse_time(c.get("publishAt"))
        key = s(c.get("key"))
        if not key or ms is None:
            continue
        tags = [s((h or {}).get("hashtag", {}).get("name")) for h in (c.get("hashtags") or [])]
        tags = [t for t in tags if t]
        pi = c.get("priceInfo") or {}
        paid = bool((num0(c.get("price")) > 0) or pi.get("isFree") is False)
        price = num0(c.get("price")) or num0(pi.get("oneshotLowestPrice") or pi.get("lowestPrice"))
        user = c.get("user") or {}
        title = s(c.get("name"))
        out.append({"creator": s(user.get("urlname")) or creator_id, "key": key, "title": title,
                    "url": s(c.get("noteUrl")) or f"https://note.com/{creator_id}/n/{key}", "publishMs": ms, "type": s(c.get("type")),
                    "paid": paid, "price": price if paid else 0, "likes": num0(c.get("likeCount")), "comments": num0(c.get("commentCount")),
                    "hashtags": tags, "imageCount": num0(c.get("imageCount")), "pinned": bool(c.get("isPinned"))})
    return {"articles": out, "isLast": bool(d.get("isLastPage")) or len(d["contents"]) == 0, "total": num0(d.get("totalCount"))}


def html_to_text(h):
    t = re.sub(r"<br\s*/?>", "\n", s(h), flags=re.I)
    t = re.sub(r"</(p|h[1-6]|li|figure|blockquote|pre)>", "\n", t, flags=re.I)
    t = re.sub(r"<[^>]+>", "", t)
    for a, b in (("&nbsp;", " "), ("&lt;", "<"), ("&gt;", ">"), ("&quot;", '"'), ("&#39;", "'"), ("&amp;", "&")):
        t = t.replace(a, b)
    return t


def parse_detail(j):
    d = (j or {}).get("data") if isinstance(j, dict) else None
    if not isinstance(d, dict) or not d.get("key"):
        raise NaError("PARSE", "記事の詳細を読み取れませんでした。")
    body = s(d.get("body"))
    text = re.sub(r"\s+", "", html_to_text(body))
    return {"key": d["key"], "textLength": len(text), "h2Count": len(re.findall(r"<h2[\s>]", body, re.I)), "imageCount": len(re.findall(r"<img[\s>]", body, re.I))}


def parse_likes(j, key):
    d = (j or {}).get("data") if isinstance(j, dict) else None
    if not isinstance(d, dict) or not isinstance(d.get("likes"), list):
        raise NaError("PARSE", "スキした人の一覧を読み取れませんでした（noteの仕様が変わった可能性があります）。")
    likes = []
    for l in d["likes"]:
        u = l.get("user") or {}
        ms = parse_time(l.get("created_at"))
        if s(u.get("urlname")) and not u.get("is_me") and ms is not None:
            likes.append({"key": key, "likedMs": ms, "urlname": s(u.get("urlname")), "nickname": s(u.get("nickname"))})
    return {"isLast": bool(j.get("is_last_page")) or len(d["likes"]) == 0, "likes": likes}


CID_RE = re.compile(r"^[A-Za-z0-9_\-]{4,40}$")


def parse_comments(j, key, own=""):
    """コメントした人。本文は読まない・返さない（誰が・いつ・どの記事に、だけ）。
    own（自分の urlname）のコメントは byOwner=True にする（ファンの数には入れない）。"""
    d = j.get("data") if isinstance(j, dict) else None
    if not isinstance(d, list):
        raise NaError("PARSE", "コメントした人の一覧を読み取れませんでした（noteの仕様が変わった可能性があります）。")
    out = []
    for c in d:
        if not isinstance(c, dict) or c.get("is_blocked"):
            continue
        u = c.get("user") or {}
        name, cid, ms = s(u.get("urlname")), s(c.get("key")), parse_time(c.get("created_at"))
        if not ID_RE.match(name) or not CID_RE.match(cid) or ms is None:
            continue
        out.append({"key": key, "cid": cid, "urlname": name, "nickname": s(u.get("nickname")), "commentedMs": ms,
                    "byOwner": bool(own) and name.lower() == own.lower(), "replied": bool(c.get("is_creator_replied"))})
    nxt = j.get("next_page")
    try:
        nxt = int(nxt) if nxt else 0
    except (TypeError, ValueError):
        nxt = 0
    return {"comments": out, "next": nxt, "total": num0(j.get("total_count"))}


# ---------- ログイン中ダッシュボード（未検証：公開されている解説記事にもとづく） ----------
def auth_response_state(code, content_type, body, location=""):
    ct = s(content_type).lower()
    b = s(body)[:200].lower()
    if code in (401, 403):
        return "invalid", f"noteから「ログインしていない／許可されていない」（{code}）と返されました"
    if 300 <= code < 400:
        path = re.sub(r"^https?://[^/]+", "", s(location)).split("?")[0]
        return "invalid", f"ログイン画面などへ転送されました（{code}" + (f" → {url_host(location)}{path}" if location else "") + "）"
    if code == 429:
        return "busy", "noteから「アクセスが多い」（429）と返されました"
    if code >= 500:
        return "error", f"noteのサーバーエラー（{code}）"
    if code == 404:
        return "error", "取得先が見つかりません（404）。noteの仕様が変わった可能性があります"
    if code >= 400:
        return "error", f"noteへのアクセスでエラー（{code}）"
    if "text/html" in ct or re.match(r"^\s*(<!doctype|<html)", b):
        return "invalid", "JSON ではなく Web ページ（HTML）が返されました。ログインが切れている可能性があります"
    return "ok", ""


def _pick(o, names):
    for n in names:
        if isinstance(o, dict) and o.get(n) not in (None, ""):
            return o[n]
    return None


def parse_stats_pv(j):
    d = (j.get("data") or j) if isinstance(j, dict) else None
    lst = _pick(d, ["note_stats", "noteStats", "notes"]) if isinstance(d, dict) else None
    if not isinstance(lst, list):
        raise NaError("PARSE", "ダッシュボード（stats/pv）の応答に記事一覧（note_stats）が見つかりませんでした。noteの仕様が変わった可能性があります。")
    items, skipped = [], 0
    for x in lst:
        key = s(_pick(x, ["key", "note_key", "noteKey"])) or key_from_url(_pick(x, ["note_url", "noteUrl", "url"]))
        pv = _pick(x, ["read_count", "readCount", "pv", "page_view_count", "pageViewCount"])
        try:
            pvn = float(pv)
        except (TypeError, ValueError):
            pvn = None
        if not key or pvn is None:
            skipped += 1
            continue
        lk, cm = _pick(x, ["like_count", "likeCount"]), _pick(x, ["comment_count", "commentCount"])
        items.append({"key": key, "title": s(_pick(x, ["name", "title"])), "pv": int(pvn) if pvn.is_integer() else pvn,
                      "likes": "" if lk is None else num0(lk), "comments": "" if cm is None else num0(cm)})
    if lst and not items:
        raise NaError("PARSE", "ダッシュボード（stats/pv）の記事データに記事キーやビュー数（read_count）が見つかりませんでした。noteの仕様が変わった可能性があります。")
    last = _pick(d, ["last_page", "lastPage", "is_last_page", "isLastPage"])
    return {"items": items, "skipped": skipped, "isLast": last is True or last == "true" or len(lst) == 0}


GQL_QUERY_FULL = ("query NaDashboard($date: Datetime!, $after: String) { dashboardSummary(unit: DAY, date: $date) { lastUpdatedAt metrics { impressionCount pageViewCount } } "
                  "dashboardNoteListConnection(unit: DAY, date: $date, order: PUBLISHED_DATE_DESC, first: 50, after: $after) { pageInfo { hasNextPage endCursor } "
                  "edges { node { note { link { absoluteUrl } } metrics { impressionCount pageViewCount likeCount commentCount salesAmount } } } } }")
GQL_QUERY_MIN = ("query NaDashboard($date: Datetime!, $after: String) { dashboardSummary(unit: DAY, date: $date) { lastUpdatedAt metrics { impressionCount pageViewCount } } "
                 "dashboardNoteListConnection(unit: DAY, date: $date, order: PUBLISHED_DATE_DESC, first: 50, after: $after) { pageInfo { hasNextPage endCursor } "
                 "edges { node { note { link { absoluteUrl } } metrics { impressionCount pageViewCount } } } } }")


def gql_body(date_str, after=None, minimal=False):
    return json.dumps({"query": GQL_QUERY_MIN if minimal else GQL_QUERY_FULL, "variables": {"date": date_str + "T00:00:00.000Z", "after": after}})


def gql_token_from_set_cookie(values):
    tok = ""
    for c in values or []:
        m = re.search(r"(?:^|[;,\s])note_gql_auth_token=([^;,\s]+)", " " + c)
        if m:
            tok = m.group(1)
    return tok


def parse_gql(j):
    if isinstance(j, dict) and isinstance(j.get("errors"), list) and j["errors"]:
        msg = " / ".join(s((e or {}).get("message")) for e in j["errors"][:2] if (e or {}).get("message"))
        err = NaError("GQL", "新ダッシュボードの応答がエラーでした：" + (msg or "理由不明")[:200])
        err.field = bool(re.search(r"cannot query field|unknown (field|argument)|undefined field", msg, re.I))
        raise err
    d = j.get("data") if isinstance(j, dict) else None
    conn = d.get("dashboardNoteListConnection") if isinstance(d, dict) else None
    if not isinstance(conn, dict) or not isinstance(conn.get("edges"), list):
        raise NaError("PARSE", "新ダッシュボードの応答に記事一覧（dashboardNoteListConnection）が見つかりませんでした。noteの仕様が変わった可能性があります。")
    items, skipped = [], 0
    o = lambda v: "" if v in (None, "") else num0(v)
    for e in conn["edges"]:
        n = (e or {}).get("node") or {}
        note = n.get("note") or {}
        m = n.get("metrics") or {}
        key = key_from_url((note.get("link") or {}).get("absoluteUrl")) or s(note.get("key"))
        if not key:
            skipped += 1
            continue
        items.append({"key": key, "pv": o(m.get("pageViewCount")), "imp": o(m.get("impressionCount")), "likes": o(m.get("likeCount")), "comments": o(m.get("commentCount")), "sales": o(m.get("salesAmount"))})
    sm = ((d.get("dashboardSummary") or {}).get("metrics") or {})
    pi = conn.get("pageInfo") or {}
    all_zero = bool(items) and all(not x["pv"] and not x["imp"] for x in items)
    return {"items": items, "skipped": skipped, "hasNext": bool(pi.get("hasNextPage")) and bool(pi.get("endCursor")), "endCursor": s(pi.get("endCursor")),
            "suspectAnonymous": (not items or all_zero) and not num0(sm.get("pageViewCount")) and not num0(sm.get("impressionCount"))}

In [ ]:
%%writefile /content/note_runner_pkg/note_runner/httpc.py
"""note への HTTP。Cookie を付けるのはここ1か所だけ・送り先が note.com のときだけ。
一時トークンは graphql.note.com だけ。間隔は3秒以上あける。403/429 は止める合図。"""
import time
import urllib.request
import urllib.error

from . import core

UA = "note-analytics-runner/1.1 (+personal use; respects robots spacing)"


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    """転送（リダイレクト）は追いかけない。ログイン画面への転送＝期限切れとして扱うため"""
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


_OPENER = urllib.request.build_opener(_NoRedirect())


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    """転送（リダイレクト）は追いかけない（ログイン画面への転送を「期限切れ」として検出するため）"""
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


def default_opener():
    return urllib.request.build_opener(_NoRedirect()).open


class Stop(Exception):
    """403/429/上限：その日の取得を止める"""
    def __init__(self, code, message):
        super().__init__(message)
        self.http = code
        self.message = message


class Client:
    def __init__(self, cookie_value=None, interval_sec=3.0, max_requests=300, sleep=time.sleep, now=None, opener=None):
        self._cookie_value = cookie_value          # 生の値は内部だけ。ログ・例外には出さない
        self.interval = max(3.0, float(interval_sec or 3.0))   # ログイン中の取得と同じく最低3秒
        self.max_requests = max_requests
        self.requests = 0
        self._last = 0.0
        self._sleep = sleep
        self._now = now or (lambda: time.time() * 1000)
        self._opener = opener or _OPENER.open
        self.token = None                           # 新ダッシュボードの一時トークン（メモリのみ）

    def _wait(self):
        if self.requests >= self.max_requests:
            raise Stop(None, f"1回の取得でアクセスできる回数の上限（{self.max_requests}回）に達したので止めました。")
        gap = self.interval - (time.time() - self._last)
        if self._last and gap > 0:
            self._sleep(gap)
        self._last = time.time()
        self.requests += 1

    def _open(self, url, headers, data=None, method="GET"):
        req = urllib.request.Request(url, data=data, method=method, headers=headers)
        try:
            resp = self._opener(req, timeout=30)
            return resp.getcode(), resp.read().decode("utf-8", "replace"), {k.lower(): v for k, v in resp.headers.items()}, resp.headers.get_all("Set-Cookie") or []
        except urllib.error.HTTPError as e:
            body = ""
            try:
                body = e.read().decode("utf-8", "replace")
            except Exception:
                pass
            return e.code, body, {k.lower(): v for k, v in (e.headers or {}).items()}, (e.headers.get_all("Set-Cookie") if e.headers else []) or []

    # ---------- 公開データ（Cookie を付けない） ----------
    def get_public(self, url):
        host = core.url_host(url)
        assert host == core.HOST_COOKIE, f"想定外のURL: {url}"
        for attempt in range(2):
            self._wait()
            code, body, _, _ = self._open(url, {"Accept": "application/json", "User-Agent": UA})
            if code == 200:
                return body
            if code == 404:
                raise core.NaError("NOTFOUND", "見つかりませんでした（IDが間違っているか、記事が削除された可能性があります）")
            if code in (403, 429):
                raise Stop(code, f"noteから「アクセスが多い／許可されていない」（{code}）と返されたので、今回の取得を止めました（{self.requests}回目）。")
            if code >= 500 and attempt == 0:
                self._sleep(10)
                continue
            raise core.NaError("HTTP", f"noteへのアクセスでエラーになりました（{code}）")
        raise core.NaError("HTTP", "noteへのアクセスに失敗しました。")

    # ---------- 認証（Cookie / Bearer を付ける唯一の場所） ----------
    def _auth(self, url, kind, method="GET", data=None, extra=None, ok_codes=(200,)):
        host = core.url_host(url)
        headers = {"Accept": "application/json", "User-Agent": UA}
        if kind == "cookie":
            if host != core.HOST_COOKIE:
                raise core.NaError("GUARD", f"安全のため止めました：Cookie は note.com 以外には送りません（送り先: {host or '不正なURL'}）。")
            if not self._cookie_value:
                raise core.NaError("NOCOOKIE", "Cookie がありません。")
            headers["Cookie"] = f"{core.COOKIE_NAME}={self._cookie_value}"
        elif kind == "bearer":
            if host != core.HOST_GQL:
                raise core.NaError("GUARD", f"安全のため止めました：一時トークンは graphql.note.com 以外には送りません（送り先: {host or '不正なURL'}）。")
            if not self.token:
                raise core.NaError("GUARD", "一時トークンがありません。")
            headers["Authorization"] = f"Bearer {self.token}"
        else:
            raise core.NaError("GUARD", "認証の種類が不明です。")
        if extra:
            headers.update(extra)
        if data is not None:
            headers["Content-Type"] = "application/json"
        self._wait()
        code, body, hdrs, setck = self._open(url, headers, data=data.encode() if isinstance(data, str) else data, method=method)
        state, reason = core.auth_response_state(code, hdrs.get("content-type", ""), body, hdrs.get("location", ""))
        if state == "ok" and code not in ok_codes and not (200 <= code < 300):
            state, reason = "error", f"想定外の応答（{code}）"
        return {"code": code, "body": body, "headers": hdrs, "set_cookie": setck, "state": state, "reason": reason}

    def stats_pv(self, page):
        return self._auth(f"{core.STATS_PV_URL}?filter=all&page={page}&sort=pv", "cookie")

    def gql_auth(self):
        r = self._auth(core.GQL_AUTH_URL, "cookie", method="POST", extra={"X-Requested-With": "XMLHttpRequest"}, ok_codes=(200, 201))
        if r["state"] == "ok":
            self.token = core.gql_token_from_set_cookie(r["set_cookie"])
        return r

    def gql(self, body):
        return self._auth(core.GQL_URL, "bearer", method="POST", data=body)

In [ ]:
%%writefile /content/note_runner_pkg/note_runner/sender.py
"""受け取り用ウェブアプリ（Apps Script の doPost）へ、合言葉で署名して送る。
Apps Script 側（src/na_receiver.js）の naRxSign_ と同じ署名方式。"""
import hashlib
import hmac
import html
import json
import secrets
import time
import urllib.request
import urllib.error

import re

from . import core

RECEIVER_RE = re.compile(r"^https://script\.google\.com/(?:a/macros/[A-Za-z0-9.\-]+|macros)/s/[A-Za-z0-9_\-]{20,200}/exec$")
REDIRECT_HOSTS = ("script.googleusercontent.com", "script.google.com")


class _GoogleOnlyRedirect(urllib.request.HTTPRedirectHandler):
    """Apps Script は POST のあと script.googleusercontent.com に転送して結果を返す。それ以外への転送は追わない"""
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if core.url_host(newurl) not in REDIRECT_HOSTS:
            raise core.NaError("RECEIVER", "受け取り先が想定外の場所へ転送しようとしたので止めました。")
        return super().redirect_request(req, fp, code, msg, headers, newurl)


def check_receiver_url(url):
    if not RECEIVER_RE.match(core.s(url)):
        raise core.NaError("CONFIG", "受け取り先のURLが Apps Script のウェブアプリ（https://script.google.com/macros/s/…/exec）ではありません。")
    return url


def sign(secret, ts, nonce, body):
    return hmac.new(secret.encode("utf-8"), f"na1\n{ts}\n{nonce}\n{body}".encode("utf-8"), hashlib.sha256).hexdigest()


def envelope(secret, body_obj, now_ms=None):
    body = json.dumps(body_obj, ensure_ascii=False, separators=(",", ":"))
    ts = int(now_ms if now_ms is not None else time.time() * 1000)
    nonce = secrets.token_urlsafe(18)[:24]
    return {"v": 1, "ts": ts, "nonce": nonce, "sig": sign(secret, ts, nonce, body), "body": body}


RETRY_HTTP = (404, 408, 429, 500, 502, 503, 504)
STATE_WAITS = (10, 30, 60)   # 設定の読み込み：失敗したら 10秒・30秒・60秒あけて最大3回やり直す（読むだけなので毎回新しい封筒）
DATA_WAITS = (10, 30)        # 書き込み：最大2回やり直す。同じ封筒（同じ nonce）をそのまま送り直す
REPLAY_CODE = "REPLAY"       # 受け取り側（v1.4.1 まで）は使用済み nonce を REPLAY で断る＝前の送信はもう届いている


class _Transient(core.NaError):
    """やり直してよい失敗（Google 側の 404/5xx、HTML の応答、通信の切断・時間切れ、BUSY）"""


def _html_summary(raw):
    """Google のエラーページ（HTML）から、見出しと本文の頭だけを短く取り出す（ログ用）"""
    t = raw or ""
    m = re.search(r"<title[^>]*>(.*?)</title>", t, re.I | re.S)
    title = html.unescape(re.sub(r"\s+", " ", m.group(1))).strip() if m else ""
    body = re.sub(r"<(script|style)\b.*?</\1>", " ", t, flags=re.I | re.S)
    body = html.unescape(re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", body))).strip()
    if title and body.startswith(title):
        body = body[len(title):].strip()
    return (title + ("：" if title and body else "") + body)[:160]


def _where(url):
    """どこが返したか（受け取り先 /exec か、結果の受け渡し先 googleusercontent か）。URL そのもの（ID・キー）は出さない"""
    host = core.url_host(url or "")
    if host == "script.googleusercontent.com":
        return "結果の受け渡し先（script.googleusercontent.com）"
    if host == "script.google.com":
        return "受け取り先（script.google.com/…/exec）"
    return "受け取り先" + (f"（{host}）" if host else "")


def wire_len(obj):
    """obj を body に入れて送ったときに、送信データ（封筒の JSON）の中で body が占める文字数。
    body は ensure_ascii=False で文字列にし、封筒は json.dumps（日本語は \\uXXXX、" と \\ はエスケープ）で送るので、その後の長さで数える"""
    return len(json.dumps(json.dumps(obj, ensure_ascii=False, separators=(",", ":")))) - 2


class Sender:
    def __init__(self, url, secret, opener=None, sleep=time.sleep, log=None):
        self.url = check_receiver_url(url)
        if not secret or len(secret) < 32:
            raise core.NaError("CONFIG", "受け取り用の合言葉（NA_RECEIVER_SECRET）がありません。")
        self.secret = secret
        self._opener = opener or urllib.request.build_opener(_GoogleOnlyRedirect()).open
        self._sleep = sleep
        self._log = log or (lambda m: None)
        self.retries = 0

    def _post_env(self, env):
        data = json.dumps(env).encode("utf-8")
        req = urllib.request.Request(self.url, data=data, method="POST", headers={"Content-Type": "application/json"})
        try:
            resp = self._opener(req, timeout=60)
            raw = resp.read().decode("utf-8", "replace")
        except urllib.error.HTTPError as e:
            try:
                raw = e.read().decode("utf-8", "replace")
            except Exception:
                raw = ""
            where = _where(getattr(e, "url", None) or (e.geturl() if hasattr(e, "geturl") else ""))
            summ = _html_summary(raw) if "<" in raw[:200] else raw[:160]
            msg = f"受け取り先がエラーを返しました（HTTP {e.code}・{where}）" + (f": {summ}" if summ else "")
            raise (_Transient if e.code in RETRY_HTTP else core.NaError)("RECEIVER", msg)
        except core.NaError:
            raise
        except (urllib.error.URLError, OSError) as e:   # 通信の切断・時間切れ（socket.timeout も OSError）
            raise _Transient("RECEIVER", f"受け取り先との通信が途中で切れました（{type(e).__name__}）。")
        try:
            out = json.loads(raw)
        except ValueError:
            summ = _html_summary(raw)
            raise _Transient("RECEIVER", "受け取り先の応答が読めませんでした（JSON ではなく HTML などが返りました）" + (f": {summ}" if summ else "") + "。デプロイのURL・アクセス権も確認してください。")
        if not isinstance(out, dict):
            raise _Transient("RECEIVER", "受け取り先の応答の形が想定と違いました。")
        if not out.get("ok"):
            code, message = out.get("code") or "RECEIVER", out.get("message") or "受け取り先に拒否されました。"
            raise (_Transient if code == "BUSY" else core.NaError)(code, message)
        return out

    def _with_retry(self, label, waits, make_env, replay_ok):
        env = make_env()
        for i in range(len(waits) + 1):
            try:
                return self._post_env(env)
            except _Transient as e:
                if i >= len(waits):
                    raise core.NaError(e.code, e.message + f"（{i + 1}回試しました）")
                self.retries += 1
                self._log(f"⚠ {label}：{e.message} → {waits[i]}秒後にもう一度送ります（{i + 2}/{len(waits) + 1}回目）")
                self._sleep(waits[i])
                if not replay_ok:
                    env = make_env()   # 読むだけの操作は新しい封筒で
            except core.NaError as e:
                if replay_ok and i > 0 and e.code == REPLAY_CODE:
                    # 前の送信は受け取り側で処理済み（応答だけが届かなかった）。二重に書かないよう、ここで成功として扱う
                    self._log(f"　{label}：前の送信がすでに受け取られていました（二重には書きません）。")
                    return {"ok": True, "replayed": True}
                raise

    def get_state(self):
        return self._with_retry("シートの設定の読み込み", STATE_WAITS, lambda: envelope(self.secret, {"kind": "note-analytics", "action": "state"}), False)

    def send_data(self, payload):
        # 同じ封筒（同じ nonce・ts・署名）を送り直す：前の送信が届いていれば受け取り側が REPLAY で断るので、記事推移などが二重にならない。
        # 時刻のずれの許容は ±10分、nonce の記録は30分なので、やり直しの待ち時間（合計40秒）はその中に収まる
        n = payload.get("chunk"), payload.get("chunks")
        label = "シートへの書き込み" + (f"（{n[0]}/{n[1]}回目）" if n[0] else "")
        return self._with_retry(label, DATA_WAITS, lambda: envelope(self.secret, payload), True)

In [ ]:
%%writefile /content/note_runner_pkg/note_runner/fetch.py
"""1日1回の取得。順番は Apps Script 版と同じ：
記事一覧・フォロワー → ベンチマーク → ダッシュボード（Cookie があるときだけ・1日1回）→ 本文の文字数 → 誰からのスキ → コメントした人（最後）。
403/429 が返ったら、その時点で note へのアクセスをやめ、取れた分と理由を記録する。"""
import json

from . import core
from .httpc import Stop


def _int(v, default, lo, hi):
    try:
        x = int(float(v))
    except (TypeError, ValueError):
        x = default
    return max(lo, min(hi, x))


def plan_settings(state, env_own="", env_bench=None):
    st = (state or {}).get("settings") or {}
    own = core.normalize_id((state or {}).get("own") or env_own)
    bench = [b for b in ((state or {}).get("bench") or env_bench or []) if core.normalize_id(b) and b != own][:5]
    return {
        "own": own, "bench": bench,
        "interval": max(3.0, float(st.get("intervalSec") or 3)),
        "ownPages": _int(st.get("ownPages") or 300, 300, 1, 300),
        "benchPages": _int(st.get("benchPages") or 3, 3, 1, 10),
        "detailMode": st.get("detailMode") if st.get("detailMode") in ("自分だけ", "すべて", "しない") else "自分だけ",
        "detailLimit": _int(st.get("detailLimit") if st.get("detailLimit") is not None else 30, 30, 0, 100),
        "snapDays": _int(st.get("snapDays") or 30, 30, 1, 3650),
        "likers": st.get("likers", True) is not False,
        "likerPages": _int(st.get("likerPages") or 5, 5, 1, 5),
        "likerArticles": 30,
        # コメントした人：シート側が v1.3.0 以上（設定に commenters がある）ときだけ。古いシートは受け取れないので取りに行かない
        "commenters": "commenters" in st and st.get("commenters") is not False,
        "commenterArticles": 20, "commenterPages": 3,
        "dash": bool(st.get("dash")), "dashImp": st.get("dashImp", True) is not False,
        "dashPages": _int(st.get("dashPages") or 30, 30, 1, 50),
        "dashShareOverride": bool(st.get("dashShareOverride")),
        "shared": bool(((state or {}).get("sharing") or {}).get("shared")),
        "lastDashDate": (state or {}).get("lastDashDate") or "",
        "known": {r[0]: {"likes": r[1], "likersAt": r[2] or 0, "hasText": bool(r[3]), "lastLike": r[4] or 0, "publishMs": r[5],
                         "commentersAt": (r[6] if len(r) > 6 else 0) or 0} for r in ((state or {}).get("ownArticles") or []) if r and r[0]},
    }


def run(http, P, now_ms, log, has_cookie=False, dash_off=False):
    out = {"profiles": [], "articles": [], "details": [], "snaps": [], "likers": [], "likersAt": [], "commenters": [], "commentersAt": [], "pv": [], "errors": [], "stopped": "", "stopHttp": None,
           "likersChecked": False, "commentersChecked": False, "dash": {"state": "none", "reason": "", "pvRows": 0, "dailyRows": 0, "notes": []}}
    today = core.jst(now_ms)["date"]
    arts = {}

    def err(msg):
        if len(out["errors"]) < 20:
            out["errors"].append(msg)

    def task(name, fn):
        if out["stopped"]:
            return
        try:
            fn()
        except Stop as e:
            out["stopped"] = f"「{name}」の取得中に：{e.message}"
            out["stopHttp"] = e.http
            log(f"止めました：{out['stopped']}")
        except core.NaError as e:
            err(f"{name}: {e.message}")
            log(f"エラー：{name}: {e.message}")

    # 1) 記事一覧・フォロワー（自分 → ベンチマーク）
    creators = ([(P["own"], True)] if P["own"] else []) + [(b, False) for b in P["bench"]]
    for cid, own in creators:
        def prof(cid=cid):
            out["profiles"].append(core.parse_creator(json.loads(http.get_public(core.creator_url(cid)))))
        task(f"プロフィール {cid}", prof)

        def lst(cid=cid, own=own):
            mx = P["ownPages"] if own else P["benchPages"]
            for page in range(1, mx + 1):
                L = core.parse_list(json.loads(http.get_public(core.list_url(cid, page))), cid)
                for a in L["articles"]:
                    if a["creator"] != cid:
                        continue   # 共同マガジンなどの他人の記事は入れない
                    arts[a["key"]] = a
                    out["articles"].append(a)
                    age_h = (now_ms - a["publishMs"]) / core.HOUR_MS
                    if own or age_h <= P["snapDays"] * 24:
                        out["snaps"].append({"creator": cid, "key": a["key"], "ageH": round(age_h, 1), "likes": a["likes"], "comments": a["comments"], "t": now_ms})
                if L["isLast"] or not L["articles"]:
                    break
        task(f"記事一覧 {cid}", lst)

    # 2) ログイン中ダッシュボード（自分の Cookie があるときだけ・1日1回）
    D = out["dash"]
    if dash_off:
        D["state"], D["reason"] = "skipped", "この実行では PV を取りませんでした（--no-dashboard）"
    elif not has_cookie:
        D["state"], D["reason"] = "none", "Cookie（NOTE_SESSION）がないので、PVの自動取得はしませんでした"
    elif not P["dash"]:
        D["state"], D["reason"] = "skipped", "シートの設定「PVの自動取得（自分のCookie）」が「いいえ」です"
    elif P["lastDashDate"] == today:
        D["state"], D["reason"] = "skipped", "今日はもう取得しています（1日1回）"
    elif P["shared"] and not P["dashShareOverride"]:
        D["state"], D["reason"] = "skipped", "スプレッドシートが自分以外と共有されているので、Cookie を使う取得はしませんでした"
    elif not P["own"]:
        D["state"], D["reason"] = "skipped", "自分のクリエイターIDがありません"
    else:
        task("PVの自動取得", lambda: _dash(http, P, now_ms, out, arts, log))

    # 3) 本文の文字数（まだ取っていない記事だけ）
    if P["detailMode"] != "しない" and P["detailLimit"] > 0:
        def det():
            cand = [a for a in arts.values() if (a["creator"] == P["own"] or P["detailMode"] == "すべて") and not P["known"].get(a["key"], {}).get("hasText")]
            cand.sort(key=lambda a: -a["publishMs"])
            for a in cand[:P["detailLimit"]]:
                try:
                    out["details"].append(core.parse_detail(json.loads(http.get_public(core.detail_url(a["key"])))))
                except core.NaError as e:
                    err(f"本文 {a['key']}: {e.message}")
        task("本文の文字数", det)

    # 4) 誰からのスキ（自分の記事だけ・最後。日付だけ記録）
    if P["likers"] and P["own"]:
        def lik():
            mine = [a for a in arts.values() if a["creator"] == P["own"] and a["likes"] > (P["known"].get(a["key"], {}).get("likersAt") or 0)]
            mine.sort(key=lambda a: -a["publishMs"])
            seen = set()
            for a in mine[:P["likerArticles"]]:
                since = P["known"].get(a["key"], {}).get("lastLike") or 0   # 前回記録した、いちばん新しいスキの日（0時）
                for page in range(1, P["likerPages"] + 1):
                    R = core.parse_likes(json.loads(http.get_public(core.likes_url(a["key"], page))), a["key"])
                    stop = R["isLast"]
                    for l in R["likes"]:
                        if l["likedMs"] < since:
                            stop = True
                            break
                        k = a["key"] + "|" + l["urlname"]
                        if k in seen:
                            continue
                        seen.add(k)
                        out["likers"].append({"key": a["key"], "urlname": l["urlname"], "nickname": l["nickname"], "day": core.jst(l["likedMs"])["date"]})
                    if stop:
                        break
                out["likersAt"].append({"key": a["key"], "likes": a["likes"]})
                out["likersChecked"] = True
        task("スキした人", lik)

    # 5) コメントした人（自分の記事だけ・スキの後＝いちばん最後。本文は保存しない。日付だけ記録）
    #    前回確認したときよりコメント数が増えた記事だけ。初回はコメントのある記事を新しい順に。1回20記事・1記事3ページまで
    if P["commenters"] and P["own"]:
        def com():
            mine = [a for a in arts.values() if a["creator"] == P["own"] and a["comments"] > (P["known"].get(a["key"], {}).get("commentersAt") or 0)]
            mine.sort(key=lambda a: -a["publishMs"])
            seen = set()
            for a in mine[:P["commenterArticles"]]:
                page = 1
                try:
                    for _ in range(P["commenterPages"]):
                        R = core.parse_comments(json.loads(http.get_public(core.comments_url(a["key"], page))), a["key"], P["own"])
                        for c in R["comments"]:
                            if c["cid"] in seen:
                                continue
                            seen.add(c["cid"])
                            out["commenters"].append({"key": a["key"], "cid": c["cid"], "urlname": c["urlname"], "nickname": c["nickname"],
                                                      "day": core.jst(c["commentedMs"])["date"], "byOwner": c["byOwner"], "replied": c["replied"]})
                        if not R["next"] or R["next"] <= page:
                            break
                        page = R["next"]
                except core.NaError as e:
                    if e.code != "NOTFOUND":
                        raise
                    err(f"コメント {a['key']}: {e.message}")   # 削除された記事など。ほかの記事は続ける
                out["commentersAt"].append({"key": a["key"], "comments": a["comments"]})
                out["commentersChecked"] = True
        task("コメントした人", com)
    return out


def _dash(http, P, now_ms, out, arts, log):
    D = out["dash"]
    today = core.jst(now_ms)["date"]
    yday = core.jst(now_ms - core.DAY_MS)["date"]

    def check(r, kind):
        if r["state"] == "ok":
            return
        if r["state"] == "invalid":
            if kind == "cookie":
                D["state"], D["reason"] = "invalid", r["reason"]
                raise core.NaError("COOKIE_INVALID", "Cookie が使えなくなりました：" + r["reason"])
            raise core.NaError("TOKEN", "新ダッシュボードの一時トークンが受け付けられませんでした：" + r["reason"])
        if r["state"] == "busy":
            raise Stop(429, r["reason"] + "。今日の取得は止めます。")
        raise core.NaError("HTTP", r["reason"])

    for page in range(1, P["dashPages"] + 1):
        r = http.stats_pv(page)
        check(r, "cookie")
        try:
            j = json.loads(r["body"])
        except ValueError:
            raise core.NaError("PARSE", "ダッシュボード（stats/pv）の応答が JSON ではありませんでした。noteの仕様が変わった可能性があります。")
        S = core.parse_stats_pv(j)
        for x in S["items"]:
            a = arts.get(x["key"])
            out["pv"].append({"date": today, "period": "全期間", "key": x["key"], "title": a["title"] if a else x["title"], "pv": x["pv"], "imp": "", "likes": x["likes"], "comments": x["comments"], "sales": "", "method": core.METHOD_STATS})
            D["pvRows"] += 1
        if S["isLast"] or not S["items"]:
            break
        if page == P["dashPages"]:
            D["notes"].append(f"stats/pv：最大ページ数（{P['dashPages']}）で止めました")
    D["state"] = "ok"
    if not P["dashImp"]:
        return
    try:
        r = http.gql_auth()
        check(r, "cookie")
        if not http.token:
            raise core.NaError("GQL", "新ダッシュボード用の一時トークンが返されませんでした（noteの仕様が変わった可能性があります）。インプレッションは取得できません。")
        after, minimal = None, False
        for page in range(1, min(10, P["dashPages"]) + 1):
            r = http.gql(core.gql_body(yday, after, minimal))
            check(r, "bearer")
            try:
                Q = core.parse_gql(json.loads(r["body"]))
            except core.NaError as e:
                if e.code == "GQL" and not minimal:
                    minimal = True
                    D["notes"].append("新ダッシュボード：スキ・コメント・売上の項目が使えなかったので、PV・インプレッションだけ取得します")
                    r = http.gql(core.gql_body(yday, after, True))
                    check(r, "bearer")
                    Q = core.parse_gql(json.loads(r["body"]))
                else:
                    raise
            except ValueError:
                raise core.NaError("PARSE", "新ダッシュボードの応答が JSON ではありませんでした。")
            if page == 1 and Q["suspectAnonymous"]:
                D["notes"].append("新ダッシュボード：数字がすべて0でした（ログインが通っていない可能性）。記録しませんでした")
                return
            for x in Q["items"]:
                a = arts.get(x["key"])
                out["pv"].append({"date": yday, "period": "日次", "key": x["key"], "title": a["title"] if a else "", "pv": x["pv"], "imp": x["imp"], "likes": x["likes"], "comments": x["comments"], "sales": x["sales"], "method": core.METHOD_GQL})
                D["dailyRows"] += 1
            if not Q["hasNext"]:
                break
            after = Q["endCursor"]
    finally:
        http.token = None   # 一時トークンはすぐ捨てる


def summary_message(out, requests):
    D = out["dash"]
    msg = f"記事 {len(out['articles'])} 件（noteへのアクセス {requests} 回）。"
    if out["stopped"]:
        msg = "途中で止めました：" + out["stopped"] + "\n" + msg
    msg += "\nPVの自動取得：" + ({"ok": f"全期間 {D['pvRows']} 記事" + (f"／前日分 {D['dailyRows']} 記事" if D["dailyRows"] else "")}.get(D["state"]) or (D["reason"] or D["state"]))
    if D["notes"]:
        msg += "\n- " + "\n- ".join(D["notes"])
    if out["likersChecked"]:
        msg += f"\n新しく記録したスキ: {len(out['likers'])} 件"
    if out["commentersChecked"]:
        msg += f"\nコメントを確認した記事: {len(out['commentersAt'])} 本（コメント {len(out['commenters'])} 件）"
    if out["errors"]:
        msg += "\nうまくいかなかったもの：\n- " + "\n- ".join(out["errors"][:5])
    return msg

In [ ]:
%%writefile /content/note_runner_pkg/note_runner/cli.py
"""実行の入口。GitHub Actions: python -m note_runner ／ Colab: note_runner.cli.colab_main()
秘密（Cookie・合言葉・受け取りURL）は環境変数か getpass でだけ受け取り、画面・ログ・ファイルには出しません。"""
import argparse
import re
import json
import os
import sys
import time
import uuid

from . import VERSION, core, fetch
from .httpc import Client
from .sender import Sender, wire_len

MAX_CHUNK = 1_000_000   # 1回に送る大きさ。実際に送る封筒の JSON（日本語は \\uXXXX にエスケープ後）の文字数で数える
RECEIVER_MAX = 1_500_000   # 受け取り側（Apps Script）の上限：封筒の JSON 全体の文字数


class Redactor:
    """ログに秘密が混ざっても伏せ字にする（念のための二重の対策）"""
    def __init__(self, secrets, stream=None):
        self.secrets = [x for x in secrets if x and len(x) >= 8]
        self.stream = stream or sys.stdout
        self.lines = []

    def clean(self, msg):
        t = str(msg)
        for x in self.secrets:
            t = t.replace(x, core.mask(x))
        return t

    def __call__(self, msg):
        t = self.clean(msg)
        self.lines.append(t)
        print(t, file=self.stream, flush=True)


REPO_RE = re.compile(r"^[A-Za-z0-9-]{1,39}/[A-Za-z0-9._-]{1,100}$")


def repo_of(env):
    """GitHub Actions で動いているときだけ「持ち主/名前」（シートのトークン登録の初期値に使う）。形が違えば送らない"""
    if (env or {}).get("GITHUB_ACTIONS") != "true":
        return ""
    r = (env.get("GITHUB_REPOSITORY") or "").strip()
    return r if REPO_RE.fullmatch(r) else ""


def chunk_payloads(out, run_id, source, started_ms, requests, message, has_cookie=False, repo=""):
    D = out["dash"]
    # PV をスキップしても Cookie があれば authState=ok。受け取り側は skipped のとき前回 cookie を引き継ぐため、
    # 以前 none のままだと「NOTE_SESSIONがありません」が残り続ける。
    auth = {"ok": "ok", "invalid": "invalid", "none": "none"}.get(D["state"], "skipped")
    if auth == "skipped" and has_cookie:
        auth = "ok"
    log = {"result": "中断" if out["stopped"] else ("一部エラー" if out["errors"] else "成功"), "requests": requests, "message": message, "startedAt": started_ms,
           "authState": auth, "authReason": D["reason"] if D["state"] == "invalid" else "", "dashOk": D["state"] == "ok"}
    base = lambda: {"kind": "note-analytics", "action": "data", "runId": run_id, "source": source}
    bodies = []
    for sec in ("articles", "snaps", "likers", "commenters", "pv"):
        cur, size = [], 0
        for item in out[sec]:
            n = wire_len(item) + 1   # エスケープ後の長さ（日本語1文字＝6文字）。エスケープ前で数えると上限を超えることがある
            if cur and size + n > MAX_CHUNK:
                b = base(); b[sec] = cur; bodies.append(b); cur, size = [], 0
            cur.append(item); size += n
        if cur:
            b = base(); b[sec] = cur; bodies.append(b)
    fin = base()
    fin.update({"profiles": out["profiles"], "details": out["details"], "likersAt": out["likersAt"], "likersChecked": out["likersChecked"],
                "commentersAt": out["commentersAt"], "commentersChecked": out["commentersChecked"], "log": log, "final": True})
    if repo and REPO_RE.fullmatch(repo):
        fin["repo"] = repo
    bodies.append(fin)
    for i, b in enumerate(bodies):
        b["chunk"], b["chunks"] = i + 1, len(bodies)
    return bodies


def too_large(bodies):
    """受け取り側の上限を超える回があれば、その番号（1から）。封筒の他の項目（署名など）の分として 1,000 文字の余裕をみる"""
    for i, b in enumerate(bodies):
        if wire_len(b) + 1000 > RECEIVER_MAX:
            return i + 1
    return 0


def run(env, argv=None, source="github", log_stream=None, opener=None, sender=None, sleep=time.sleep, now_ms=None):
    ap = argparse.ArgumentParser(prog="note_runner")
    ap.add_argument("--dry-run", action="store_true", help="シートに送らず、取得結果をファイルに書く（--out）")
    ap.add_argument("--out", default=os.path.join(os.environ.get("RUNNER_TEMP") or "/tmp", "note-runner-dry-run.json"))
    ap.add_argument("--no-dashboard", action="store_true", help="Cookie があっても PV の自動取得をしない")
    a = ap.parse_args(argv or [])

    cookie_raw = env.get("NOTE_SESSION") or ""
    cookie, why = (None, "") if not cookie_raw else core.normalize_cookie(cookie_raw)
    url, secret = env.get("NA_RECEIVER_URL") or "", env.get("NA_RECEIVER_SECRET") or ""
    secrets_list = [cookie_raw, cookie or "", secret, url.split("/s/")[-1].split("/")[0] if "/s/" in url else ""]
    if env.get("GITHUB_ACTIONS") == "true":
        for x in secrets_list:
            if x:
                print(f"::add-mask::{x}", file=log_stream or sys.stdout, flush=True)   # GitHub のログで自動的に *** に置き換えさせる
    log = Redactor(secrets_list, log_stream)
    log(f"note分析ランナー v{VERSION}（{source}）")
    if cookie_raw and not cookie:
        log("⚠ NOTE_SESSION の形が想定と違うので、PVの自動取得はしません：" + why)
    if a.no_dashboard:
        cookie = None

    out_path = os.path.abspath(a.out)
    here = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
    if a.dry_run and (out_path == here or out_path.startswith(here + os.sep)):
        log("✗ 取得データをリポジトリの中には書きません（--out をリポジトリの外にしてください）。")
        return 1, log
    snd = sender
    state = {}
    if not a.dry_run or (url and secret):   # お試しでも、受け取り先があれば設定だけ読む（書き込みはしない）
        try:
            snd = snd or Sender(url, secret, opener=opener, sleep=sleep, log=log)
            state = snd.get_state()
        except core.NaError as e:
            log("✗ シートの受け取り先に接続できませんでした：" + e.message)
            return 1, log
        # no_dashboard のときは Cookie をわざと使わないので「なし」とは書かない（誤解を防ぐ）
        cookie_txt = ('あり ' + core.mask(cookie)) if cookie else ('この実行では使わない（--no-dashboard）' if a.no_dashboard else 'なし')
        log(f"シートの設定を読みました：自分={state.get('own') or '（なし）'}／ベンチマーク={','.join(state.get('bench') or []) or '（なし）'}／Cookie={cookie_txt}")
    P = fetch.plan_settings(state, env.get("NOTE_OWN", ""), [b for b in (env.get("NOTE_BENCH") or "").split(",") if b])
    if not P["own"] and not P["bench"]:
        log("✗ 自分のクリエイターIDもベンチマークもありません（シートの「設定」を確認してください）。")
        return 1, log
    started = int(now_ms if now_ms is not None else time.time() * 1000)
    http = Client(cookie_value=cookie, interval_sec=P["interval"], sleep=sleep, opener=opener)
    out = fetch.run(http, P, started, log, has_cookie=bool(cookie), dash_off=bool(a.no_dashboard))
    msg = fetch.summary_message(out, http.requests)
    bodies = chunk_payloads(out, uuid.uuid4().hex[:12], source, started, http.requests, msg, has_cookie=bool(cookie), repo=repo_of(env))
    # 念のため：送るデータに Cookie・トークンが絶対に入っていないこと
    blob = json.dumps(bodies, ensure_ascii=False)
    for x in [cookie_raw, cookie, http.token, secret]:
        if x and x in blob:
            log("✗ 安全のため止めました：送るデータに秘密の文字列が入っていました。")
            return 1, log
    big = too_large(bodies)
    if big:
        log(f"✗ 送るデータ（{big}回目）が受け取り側の上限（{RECEIVER_MAX:,} 文字）を超えるので、送らずに止めました。")
        return 1, log
    if a.dry_run:
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(bodies, f, ensure_ascii=False)
        log(f"（お試し）シートには送らず {out_path} に書きました（{len(bodies)} 回分）。")
    else:
        try:
            for b in bodies:
                snd.send_data(b)
        except core.NaError as e:
            log("✗ シートへの書き込みで止まりました：" + e.message)
            return 1, log
        again = getattr(snd, "retries", 0)
        log(f"シートに送りました（{len(bodies)} 回に分けて）。" + (f"（途中で {again} 回送り直しました）" if again else ""))
    log(msg)
    if out["stopped"]:
        log(("::warning::" if env.get("GITHUB_ACTIONS") == "true" else "⚠ ") + "noteへのアクセスを途中で止めました：" + out["stopped"])
    if out["dash"]["state"] == "invalid":
        log(("::error::" if env.get("GITHUB_ACTIONS") == "true" else "✗ ") + "note の Cookie が使えなくなりました。Secrets の NOTE_SESSION を新しい値にしてください。")
        return 1, log
    return 0, log


def main():
    code, _ = run(dict(os.environ), sys.argv[1:], source=os.environ.get("NA_SOURCE", "github"))
    sys.exit(code)


def colab_main(dry_run=False, use_cookie=True):
    """Colab 用。入力はすべて getpass（画面にもノートブックにも残らない）"""
    from getpass import getpass
    env = {}
    env["NA_RECEIVER_URL"] = input("受け取り用ウェブアプリのURL（https://script.google.com/macros/s/…/exec）: ").strip() if not dry_run else ""
    env["NA_RECEIVER_SECRET"] = getpass("受け取り用の合言葉（入力しても表示されません）: ").strip() if not dry_run else ""
    if use_cookie:
        env["NOTE_SESSION"] = getpass("note の _note_session_v5 の値（使わないなら空のまま Enter）: ").strip()
    if dry_run:
        env["NOTE_OWN"] = input("自分のクリエイターID: ").strip()
    try:
        code, _ = run(env, ["--dry-run"] if dry_run else [], source="colab")
    finally:
        env.clear()   # 入力した値をすぐ消す
    return code

## 2. 実行する

- `dry_run=True` にすると、シートに送らずに取得だけを試します（結果は Colab の一時フォルダ `/tmp` に置き、すぐ消えます）。
- `use_cookie=False` にすると、Cookie を使わず公開データだけを取ります（PV の自動取得はしません）。

In [ ]:
import sys, importlib
sys.path.insert(0, '/content/note_runner_pkg')
import note_runner.cli as cli
importlib.reload(cli)
cli.colab_main(dry_run=False, use_cookie=True)

## 3. 終わったら

- 「ランタイム → ランタイムを接続解除して削除」を押すと、この実行環境ごと消えます（入力した値も残りません）。
- 結果はシートの「取得ログ」と、スマホ画面の「GitHub/Colab」表示で確認できます。